# QC-Py-Cloud-03 : Parite de Risque (Risk Parity)

[< Classification de Texte](./QC-Py-Cloud-02-ML-Classification.ipynb) | [Suivant : RL DQN >](./QC-Py-Cloud-04-RL-DQN-Trading.ipynb)

**Niveau** : Intermediaire | **Duree estimee** : 25 min | **Kernel** : Python 3

## Objectifs

- Comprendre le concept de parite de risque (Risk Parity / Risk Budgeting)
- Implementer la ponderation par volatilite inverse (Inverse Volatility Weighting)
- Gerer le rebalancement mensuel avec surveillance de derive
- Deployer sur QC Cloud et comparer avec un portefeuille equally-weighted

---

## Partie 1 : Theorie du Risk Parity

Le Risk Parity est une approche d'allocation developpee par Bridgewater Associates
(Ray Dalio, 1996). Au lieu d'allouer le capital equitablement (1/N), on alloue
le **risque** equitablement entre les classes d'actifs.

### Principe : Inverse Volatility Weighting

Pour N actifs, le poids de l'actif i est :

```
w_i = (1/sigma_i) / sum(1/sigma_j)
```

Ou sigma_i est la volatilite realisee de l'actif i sur une fenetre glissante.

In [1]:
# Demonstration : calcul de poids Risk Parity
import numpy as np

assets = {"SPY": 16.5, "EFA": 18.2, "GLD": 15.8, "DBC": 22.1, "TLT": 12.4}

n = len(assets)
equal_weights = {k: 1/n for k in assets}

inv_vols = {k: 1/v for k, v in assets.items()}
total = sum(inv_vols.values())
rp_weights = {k: v/total for k, v in inv_vols.items()}

print("Allocation :  Equal-Weight  vs  Risk Parity")
print("-" * 45)
for asset in assets:
    ew = equal_weights[asset]
    rp = rp_weights[asset]
    print(f"  {asset:5s} :    {ew:5.1%}         {rp:5.1%}")

print()
ew_risk = {k: equal_weights[k] * assets[k] for k in assets}
rp_risk = {k: rp_weights[k] * assets[k] for k in assets}
print("Contribution au risque (%):")
print("-" * 45)
for asset in assets:
    print(f"  {asset:5s} :    {ew_risk[asset]:5.1f}%         {rp_risk[asset]:5.1f}%")


Allocation :  Equal-Weight  vs  Risk Parity
---------------------------------------------
  SPY   :    20.0%         19.9%
  EFA   :    20.0%         18.0%
  GLD   :    20.0%         20.8%
  DBC   :    20.0%         14.8%
  TLT   :    20.0%         26.5%

Contribution au risque (%):
---------------------------------------------
  SPY   :      3.3%           3.3%
  EFA   :      3.6%           3.3%
  GLD   :      3.2%           3.3%
  DBC   :      4.4%           3.3%
  TLT   :      2.5%           3.3%


La contribution au risque du portefeuille Risk Parity est beaucoup plus equilibree
que celle du portefeuille equally-weighted.

### Parametres de la strategie QC

| ETF | Classe | Role dans le portefeuille |
|-----|--------|---------------------------|
| SPY | Actions US | Growth engine |
| EFA | Actions internationales | Diversification geographique |
| GLD | Or | Couverture inflation |
| DBC | Matieres premieres | Exposition reelle |
| TLT | Obligations long terme | Safe haven / stabilisateur |

Rebalancement mensuel avec trigger de derive a 5% (intra-mensuel).

In [2]:
# Code source de l'algorithme - pret pour deploiement QC Cloud
qc_code = '''# region imports
from AlgorithmImports import *
# endregion


class RiskParity(QCAlgorithm):
    """
    Risk Parity Strategy (Bridgewater-style simplifie).

    Edge: Equaliser la contribution au risque de chaque classe d'actifs
    plutot que la contribution en capital.
    Reference: Asness/Frazzini 2012 "Leverage Aversion and Risk Parity"

    Universe: SPY, EFA, GLD, DBC, TLT (5 classes d'actifs diversifiees)
    Signal: Poids = 1/volatilite(60j), normalises pour sommer a 1.0
    Rebalancement: Mensuel + trigger 5% de derive
    Periode: 2015-2026
    """

    # Parametres configurables
    TICKERS = ["SPY", "EFA", "GLD", "DBC", "TLT"]
    VOL_LOOKBACK = 60       # Jours pour la volatilite realisee
    REBALANCE_DAY = 1       # Jour du mois pour rebalancement mensuel
    DRIFT_THRESHOLD = 0.05  # Seuil de derive pour rebalancement intra-mensuel
    WARMUP_DAYS = 70        # Jours de warmup pour avoir 60j de donnees

    def initialize(self):
        self.set_start_date(2015, 1, 1)
        self.set_cash(100000)

        # Ajouter les ETFs en resolution quotidienne (suffisant pour mensuel)
        self.symbols = {}
        for ticker in self.TICKERS:
            symbol = self.add_equity(ticker, Resolution.DAILY).symbol
            self.symbols[ticker] = symbol

        # Indicateurs de volatilite (ecart-type 60j des rendements log)
        self.std_indicators = {}
        for ticker, symbol in self.symbols.items():
            # STD sur les log-returns: utiliser StandardDeviation sur les prix
            # On calculera manuellement via RollingWindow
            self.std_indicators[ticker] = self.STD(symbol, self.VOL_LOOKBACK, Resolution.DAILY)

        # Poids cibles actuels
        self.target_weights = {ticker: 1.0 / len(self.TICKERS) for ticker in self.TICKERS}

        # Warmup pour remplir les indicateurs
        self.set_warm_up(self.WARMUP_DAYS)

        # Scheduler: rebalancement mensuel le 1er jour de trading du mois
        self.schedule.on(
            self.date_rules.month_start("SPY"),
            self.time_rules.after_market_open("SPY", 30),
            self.rebalance
        )

        self.log("RiskParity initialized: 5 ETFs, 60-day vol, monthly rebalance")

    def on_data(self, data: Slice):
        """Verifier la derive des poids intra-mensuelle."""
        if self.is_warming_up:
            return

        if not self._indicators_ready():
            return

        # Verifier si les poids actuels ont derive au-dela du seuil
        if self._check_drift():
            self.log(f"Drift threshold exceeded, rebalancing intra-month")
            self._execute_rebalance()

    def rebalance(self):
        """Rebalancement mensuel principal."""
        if self.is_warming_up:
            return

        if not self._indicators_ready():
            self.log("Indicators not ready, skipping rebalance")
            return

        self._compute_target_weights()
        self._execute_rebalance()

    def _indicators_ready(self) -> bool:
        """Verifie que tous les indicateurs sont prets."""
        return all(ind.is_ready for ind in self.std_indicators.values())

    def _compute_target_weights(self):
        """Calcule les poids inversement proportionnels a la volatilite."""
        # Recuperer la volatilite de chaque actif (STD des prix)
        # La STD des prix n'est pas la bonne mesure - on veut la vol des rendements
        # On approxime: vol_rendements ~= STD(prix) / prix_moyen
        # Mais QuantConnect STD donne directement la STD des valeurs brutes
        # Pour avoir la vol des rendements: utiliser le ratio STD/prix actuel

        vols = {}
        for ticker, symbol in self.symbols.items():
            std_val = self.std_indicators[ticker].current.value
            current_price = self.securities[symbol].price

            if current_price > 0 and std_val > 0:
                # Volatilite relative (rendements)
                vols[ticker] = std_val / current_price
            else:
                vols[ticker] = 0.01  # Valeur par defaut si donnee manquante

        # Poids = 1/vol (inverse volatility weighting)
        inv_vols = {}
        for ticker, vol in vols.items():
            if vol > 0:
                inv_vols[ticker] = 1.0 / vol
            else:
                inv_vols[ticker] = 0.0

        total_inv_vol = sum(inv_vols.values())

        if total_inv_vol > 0:
            self.target_weights = {
                ticker: inv_vol / total_inv_vol
                for ticker, inv_vol in inv_vols.items()
            }
        else:
            # Fallback: poids egaux
            n = len(self.TICKERS)
            self.target_weights = {ticker: 1.0 / n for ticker in self.TICKERS}

        # Log les poids calcules
        weight_str = ", ".join(
            f"{t}:{w:.1%}" for t, w in sorted(self.target_weights.items())
        )
        self.log(f"Target weights: {weight_str}")

    def _check_drift(self) -> bool:
        """Verifie si un actif a derive au-dela du seuil par rapport a son poids cible."""
        if not self.portfolio.invested:
            return False

        total_value = self.portfolio.total_portfolio_value
        if total_value <= 0:
            return False

        for ticker, symbol in self.symbols.items():
            target = self.target_weights.get(ticker, 0.0)
            holding = self.portfolio[symbol]
            current_weight = holding.holdings_value / total_value

            if abs(current_weight - target) > self.DRIFT_THRESHOLD:
                return True

        return False

    def _execute_rebalance(self):
        """Execute le rebalancement vers les poids cibles."""
        for ticker, symbol in self.symbols.items():
            weight = self.target_weights.get(ticker, 0.0)
            self.set_holdings(symbol, weight)

        self.log(
            f"Rebalanced | "
            f"Portfolio: ${self.portfolio.total_portfolio_value:,.0f}"
        )

'''

print(f"Algorithme charge : {len(qc_code)} caracteres")
print(f"Lignes de code : {len(qc_code.split(chr(10)))}")
print("Classe : RiskParity")
print("Univers : SPY, EFA, GLD, DBC, TLT")
print("Lookback volatilite : 60 jours")
print("Rebalancement : mensuel + derive 5%")


Algorithme charge : 5936 caracteres
Lignes de code : 163
Classe : RiskParity
Univers : SPY, EFA, GLD, DBC, TLT
Lookback volatilite : 60 jours
Rebalancement : mensuel + derive 5%


### Architecture de l'algorithme

| Composant | Methode | Description |
|-----------|---------|-------------|
| Indicateurs | `STD(60)` | Volatilite 60j par ETF |
| Poids cibles | `_compute_target_weights()` | 1/vol normalise |
| Detection derive | `_check_drift()` | Seuil 5% vs poids cible |
| Execution | `_execute_rebalance()` | `SetHoldings()` vers cibles |
| Warmup | `SetWarmUp(70)` | Remplissage initial indicateurs |

---

## Partie 2 : Deploiement via MCP QuantConnect

In [3]:
# Workflow MCP pour le Risk Parity
print("Workflow de deploiement Risk Parity :")
print()
print("1. create_project(name='RiskParity-Cloud')")
print("2. update_file_contents(projectId, name='main.py', content=qc_code)")
print("3. create_compile(projectId) -> compileId")
print("4. create_backtest(projectId, compileId, name='RiskParity-v1')")
print("5. read_backtest(projectId, backtestId)")
print()
print("Metriques attendues (estimation) :")
print("  - Sharpe : 0.6-0.8 (diversification ameliore le ratio)")
print("  - CAGR : 6-8% (sans leverage)")
print("  - MaxDD : 15-20% (protection obligations)")
print()
print("Note : Deployez via MCP pour obtenir les resultats reels.")


Workflow de deploiement Risk Parity :

1. create_project(name='RiskParity-Cloud')
2. update_file_contents(projectId, name='main.py', content=qc_code)
3. create_compile(projectId) -> compileId
4. create_backtest(projectId, compileId, name='RiskParity-v1')
5. read_backtest(projectId, backtestId)

Metriques attendues (estimation) :
  - Sharpe : 0.6-0.8 (diversification ameliore le ratio)
  - CAGR : 6-8% (sans leverage)
  - MaxDD : 15-20% (protection obligations)

Note : Deployez via MCP pour obtenir les resultats reels.


---

## Partie 3 : Resultats et interpretation

### Comparaison avec les benchmarks

| Strategie | Sharpe | CAGR | MaxDD |
|-----------|--------|------|-------|
| 60/40 (SPY/TLT) | ~0.6 | ~8% | ~25% |
| Risk Parity (sans leverage) | ~0.7 | ~7% | ~18% |
| Risk Parity (1.5x leverage) | ~0.75 | ~10% | ~25% |

### Points d'attention

- Sans leverage, le Risk Parity a un rendement inferieur au 60/40
- La derive des poids necessite une surveillance (trigger 5%)
- Les correlations entre classes d'actifs changent en periode de stress
- La fenetre de volatilite (60j) est un parametre sensible

In [4]:
# Placeholder pour les resultats du backtest QC Cloud
print("Resultats du backtest Risk Parity (a deployer via MCP) :")
print()
print("  Algorithme    : RiskParity")
print("  Periode       : 2015-01-01 -> 2026-01-01")
print("  Capital initial : 100 000 USD")
print("  Univers        : SPY, EFA, GLD, DBC, TLT")
print("  Rebalancement  : mensuel + derive 5%")
print()
print("  Metriques a recuperer via read_backtest :")
print("    - Sharpe Ratio")
print("    - Compounding Annual Return")
print("    - Max Drawdown")
print("    - Portfolio Turnover")
print()
print("Note : Deployez via MCP pour obtenir les resultats reels.")


Resultats du backtest Risk Parity (a deployer via MCP) :

  Algorithme    : RiskParity
  Periode       : 2015-01-01 -> 2026-01-01
  Capital initial : 100 000 USD
  Univers        : SPY, EFA, GLD, DBC, TLT
  Rebalancement  : mensuel + derive 5%

  Metriques a recuperer via read_backtest :
    - Sharpe Ratio
    - Compounding Annual Return
    - Max Drawdown
    - Portfolio Turnover

Note : Deployez via MCP pour obtenir les resultats reels.


---

## Conclusion

Le Risk Parity est une strategie fondamentale en gestion quantitative :
- **Principe** : allouer le risque, pas le capital
- **Implementation** : poids proportionnels a 1/volatilite
- **Avantage** : meilleure diversification du risque entre classes d'actifs
- **Limite** : sans leverage, rendement inferieur au 60/40

| Concept | Outil QC | Utilisation |
|---------|----------|-------------|
| Volatilite | `STD(60)` | Indicateur QC natif |
| Poids 1/vol | `_compute_target_weights()` | Allocation dynamique |
| Derive | `_check_drift()` | Rebalancement intra-mensuel |
| Execution | `SetHoldings()` | Rebalancement vers cibles |

[< Classification de Texte](./QC-Py-Cloud-02-ML-Classification.ipynb) | [Suivant : RL DQN >](./QC-Py-Cloud-04-RL-DQN-Trading.ipynb)